In [8]:
import pandas as pd
import numpy as np
import os
import re

In [9]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [11]:
input_path = os.path.join("..", "data", "raw", "Claims_raw_dataset.xlsx")
df = pd.read_excel(input_path)

In [12]:
# View dataset information

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 25 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   Claim Number                    1000 non-null   object        
 1   Policy Number                   1000 non-null   object        
 2   Date of Loss                    1000 non-null   datetime64[ns]
 3   Time of Loss                    1000 non-null   object        
 4   Loss Location                   1000 non-null   object        
 5   Claimant Name                   1000 non-null   object        
 6   Vehicle Make                    1000 non-null   object        
 7   Vehicle Model                   1000 non-null   object        
 8   Vehicle Year                    1000 non-null   int64         
 9   Damage Description              1000 non-null   object        
 10  Reported By                     1000 non-null   object        
 11  Claim

In [13]:
# Checking Data types
df.dtypes

Claim Number                              object
Policy Number                             object
Date of Loss                      datetime64[ns]
Time of Loss                              object
Loss Location                             object
Claimant Name                             object
Vehicle Make                              object
Vehicle Model                             object
Vehicle Year                               int64
Damage Description                        object
Reported By                               object
Claim Status                              object
Police Report                             object
Photos/Videos                             object
Repair Estimate                           object
Towing Receipt                            object
Rental Receipt                            object
Medical & Injury Documentation            object
Medical Reports                           object
Hospital Records                          object
Third-Party Informat

In [14]:
# Convert date column

df["Date of Loss"] = pd.to_datetime(df["Date of Loss"], errors="coerce")

In [15]:
# Extracting City & State from Loss Location

last_line = df["Loss Location"].str.split("\n").str[-1]

## Extract state codes
df["State"] = last_line.str.extract(r"\b([A-Z]{2})\b")

## Extract City Names
df["City"] = last_line.str.extract(r"^(.+?)(?=,|\s[A-Z]{2})")

## Handling Military Address
df.loc[last_line.str.contains("DPO|APO|FPO"), "City"] = "Military Post Office"

In [16]:
# Check missing values

df.isnull().sum()

Claim Number                        0
Policy Number                       0
Date of Loss                        0
Time of Loss                        0
Loss Location                       0
Claimant Name                       0
Vehicle Make                        0
Vehicle Model                       0
Vehicle Year                        0
Damage Description                  0
Reported By                         0
Claim Status                        0
Police Report                       0
Photos/Videos                       0
Repair Estimate                     0
Towing Receipt                      0
Rental Receipt                      0
Medical & Injury Documentation      0
Medical Reports                   800
Hospital Records                  800
Third-Party Information             0
Third-Party Insurance             750
Third-Party Claim Form              0
Repair Estimate Cost              486
Hospital Cost                     800
State                               0
City        

### Repair cost  Analysis Starting

In [ ]:
#df["Repair Estimate Cost"].isna().mean() * 100

np.float64(48.6)

In [ ]:
## Overall variance
# overall_variance = df["Repair Estimate Cost"].var()
# print("Overall Variance:", overall_variance)

Overall Variance: 1619669.876169022


In [ ]:
## Domain knowledge groups order
# groups = [
#     ["Vehicle Make","Vehicle Model","Vehicle Year","Damage Description"],
#     ["Vehicle Make","Vehicle Model","Damage Description"],
#     ["Vehicle Make","Damage Description"],
#     ["Vehicle Make","Vehicle Model","Vehicle Year"],
#     ["Vehicle Make","Vehicle Model"],
#     ["Vehicle Make"]
# ]

In [ ]:
# import pandas as pd

# results = []

# for g in groups:
#     var = df.groupby(g)["Repair Estimate Cost"].var().mean()
#     results.append({
#         "Grouping": " + ".join(g),
#         "Average Variance": var
#     })

# variance_table = pd.DataFrame(results).sort_values("Average Variance")
# print(variance_table)

                                            Grouping  Average Variance
3        Vehicle Make + Vehicle Model + Vehicle Year      1.578793e+06
4                       Vehicle Make + Vehicle Model      1.593580e+06
1  Vehicle Make + Vehicle Model + Damage Description      1.596198e+06
5                                       Vehicle Make      1.624600e+06
2                  Vehicle Make + Damage Description      1.630417e+06
0  Vehicle Make + Vehicle Model + Vehicle Year + ...      1.853486e+06


In [ ]:
### Checking group size for each of the group
# for g in groups:
#     sizes = df.groupby(g).size()
#     print("Grouping:", g)
#     print("Median group size:", sizes.median())
#     print("Groups ≥5 records:", (sizes>=5).mean())
#     print()

Grouping: ['Vehicle Make', 'Vehicle Model', 'Vehicle Year', 'Damage Description']
Median group size: 1.0
Groups ≥5 records: 0.0

Grouping: ['Vehicle Make', 'Vehicle Model', 'Damage Description']
Median group size: 5.0
Groups ≥5 records: 0.6388888888888888

Grouping: ['Vehicle Make', 'Damage Description']
Median group size: 34.5
Groups ≥5 records: 1.0

Grouping: ['Vehicle Make', 'Vehicle Model', 'Vehicle Year']
Median group size: 2.0
Groups ≥5 records: 0.022099447513812154

Grouping: ['Vehicle Make', 'Vehicle Model']
Median group size: 28.0
Groups ≥5 records: 1.0

Grouping: ['Vehicle Make']
Median group size: 166.5
Groups ≥5 records: 1.0



In [22]:
### Hierarchical Imputation (Repair Cost) - Cascading Fallback Strategy

group_levels = [
    #["Vehicle Make","Vehicle Model","Vehicle Year"],
    ["Vehicle Make","Vehicle Model"],
    ["Vehicle Make","Vehicle Model","Damage Description"],
    ["Vehicle Make"],
    ["Vehicle Make","Damage Description"]
]

for cols in group_levels:
    df["Repair Estimate Cost"] = df["Repair Estimate Cost"].fillna(
        df.groupby(cols)["Repair Estimate Cost"].transform("median")
    )

df["Repair Estimate Cost"] = df["Repair Estimate Cost"].fillna(
    df["Repair Estimate Cost"].median()
)

In [23]:
# Check missing values

df.isnull().sum()

Claim Number                        0
Policy Number                       0
Date of Loss                        0
Time of Loss                        0
Loss Location                       0
Claimant Name                       0
Vehicle Make                        0
Vehicle Model                       0
Vehicle Year                        0
Damage Description                  0
Reported By                         0
Claim Status                        0
Police Report                       0
Photos/Videos                       0
Repair Estimate                     0
Towing Receipt                      0
Rental Receipt                      0
Medical & Injury Documentation      0
Medical Reports                   800
Hospital Records                  800
Third-Party Information             0
Third-Party Insurance             750
Third-Party Claim Form              0
Repair Estimate Cost                0
Hospital Cost                     800
State                               0
City        

In [24]:
## Checking NULLS for Hospital Cost
df.loc[(df["Medical & Injury Documentation"] == "YES") &(df["Hospital Cost"].isna())].head()

,Claim Number,Policy Number,Date of Loss,Time of Loss,Loss Location,Claimant Name,Vehicle Make,Vehicle Model,Vehicle Year,Damage Description,Reported By,Claim Status,Police Report,Photos/Videos,Repair Estimate,Towing Receipt,Rental Receipt,Medical & Injury Documentation,Medical Reports,Hospital Records,Third-Party Information,Third-Party Insurance,Third-Party Claim Form,Repair Estimate Cost,Hospital Cost,State,City


In [25]:
## Checking NULLS for Third-Party Insurance
df.loc[(df["Third-Party Information"] == "YES") & (df["Third-Party Insurance"].isna())].head()

,Claim Number,Policy Number,Date of Loss,Time of Loss,Loss Location,Claimant Name,Vehicle Make,Vehicle Model,Vehicle Year,Damage Description,Reported By,Claim Status,Police Report,Photos/Videos,Repair Estimate,Towing Receipt,Rental Receipt,Medical & Injury Documentation,Medical Reports,Hospital Records,Third-Party Information,Third-Party Insurance,Third-Party Claim Form,Repair Estimate Cost,Hospital Cost,State,City


In [26]:
df.head()

,Claim Number,Policy Number,Date of Loss,Time of Loss,Loss Location,Claimant Name,Vehicle Make,Vehicle Model,Vehicle Year,Damage Description,Reported By,Claim Status,Police Report,Photos/Videos,Repair Estimate,Towing Receipt,Rental Receipt,Medical & Injury Documentation,Medical Reports,Hospital Records,Third-Party Information,Third-Party Insurance,Third-Party Claim Form,Repair Estimate Cost,Hospital Cost,State,City
0,C9CB6205,6B2AECF6-1,2023-05-07,04:03:07,"37802 Andre Shoals Suite 964\nSouth Amanda, KY...",George Norman,Honda,F-150,2006,Side collision,Courtney Richardson,Under Investigation,Yes,No,No,No,No,Yes,Fractured arm,Surgery required,Yes,Conner-Coleman,Yes,2611.0,5892.0,KY,South Amanda
1,A8538ED6,9E4F07E8-C,2025-02-05,07:43:50,"88511 John Mission\nEast Kylebury, OR 94066",Kendra Jenkins,BMW,Camry,2006,Front-end damage,Julie Browning,Pending,Yes,No,Yes,Yes,No,Yes,Whiplash,ER visit,Yes,"Shelton, Ware and Sanchez",Yes,3888.0,5219.0,OR,East Kylebury
2,417ECBC3,ECBCBF34-1,2024-06-13,17:12:02,"031 Campbell Passage Suite 918\nPalmerland, NC...",Dawn Turner,Toyota,F-150,2020,Minor scratches,Kathleen Garcia,Pending,No,No,Yes,Yes,Yes,Yes,Minor cuts and bruises,Physiotherapy,Yes,Williams LLC,Yes,2660.0,4190.0,NC,Palmerland
3,E69D0F55,E9375C38-3,2024-04-29,02:40:26,Unit 2481 Box 9128\nDPO AE 16702,Jaime Johnson,Toyota,C-Class,2012,Side collision,Erin Parks,Open,Yes,Yes,Yes,No,No,Yes,Fractured arm,Outpatient consultation,Yes,Erickson-Cunningham,Yes,3458.0,8309.0,AE,Military Post Office
4,996A769D,61F03056-7,2022-12-28,10:56:43,2043 Jeremy Estates Apt. 970\nPort Jenniferber...,Brooke Greer,BMW,C-Class,2012,Side collision,Thomas Lopez,Under Investigation,Yes,Yes,No,Yes,Yes,Yes,Internal injuries,Overnight observation,Yes,"Taylor, Knight and Jackson",Yes,1681.5,9774.0,WI,Port Jenniferberg


In [ ]:
output_path = "../data/processed/Cleaned_Processed_Claims_dataset.xlsx"

df.to_excel(output_path, index=False)

print("Clean dataset saved successfully")

Clean dataset saved successfully
